# Week 62, Day 6: Forge: Week 62 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w062/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
print("1.", 10 * 2 ** 2)
print("2.", 40 * 39 // 2)
print("3. about", 99 * 4 * 4, "(one 4-by-4 matrix-vector product per step)")

### Exercise: Part A: a shared input defeats the pairwise model

Simulate the data, fit the pairwise model by moment matching as on Day 5, and compare how the data, the independent model and the pairwise model distribute the number of neurons active in a bin, in particular the probability of 4 to 6 and of 9 or 10 active.

In [ ]:
import numpy as np
import itertools
N = 10
W = np.array(list(itertools.product((0, 1), repeat=N)), dtype=float)

def model_probs(h, J):
    E = W @ h + 0.5 * np.einsum("ki,ij,kj->k", W, J, W)
    p = np.exp(E - E.max())
    return p / p.sum()

gA = np.random.default_rng(626)
T = 40000
up = gA.random(T) < 0.1                                  # shared input high in 10% of bins
p_fire = np.where(up[:, None], 0.55, 0.05) * gA.uniform(0.8, 1.2, N)[None, :]
words = (gA.random((T, N)) < p_fire).astype(float)
idx = (words @ (2 ** np.arange(N - 1, -1, -1))).astype(int)
p_emp = np.bincount(idx, minlength=2 ** N) / T
rates = words.mean(0); M2 = words.T @ words / T

h_fit = np.log(rates / (1 - rates)); J_fit = np.zeros((N, N))
for it in range(3000):
    p = model_probs(h_fit, J_fit)
    h_fit += ...
    dJ = M2 - (W * p[:, None]).T @ W; np.fill_diagonal(dJ, 0)
    J_fit += dJ
p_pair = model_probs(h_fit, J_fit)
p_ind = np.prod(np.where(W == 1, rates, 1 - rates), axis=1)
k_on = W.sum(1)
mid_emp, mid_ind, mid_pair = (p[(k_on >= 4) & (k_on <= 6)].sum() for p in (p_emp, p_ind, p_pair))
top_emp, top_ind, top_pair = (p[k_on >= 9].sum() for p in (p_emp, p_ind, p_pair))
print(f"P(4 to 6 active): data {mid_emp:.4f}, independent {mid_ind:.2e}, pairwise {mid_pair:.4f}")
print(f"P(9 or 10 active): data {top_emp:.4f}, independent {top_ind:.2e}, pairwise {top_pair:.4f}")
print("P(K neurons firing), K = 0 to 10:")
for name, p in [("data", p_emp), ("pairwise", p_pair)]:
    print(f"  {name:9s}", np.round([p[k_on == k].sum() for k in range(11)], 4))

In [ ]:
import numpy as np
def test_fit():
    "the pairwise model matches the data's rates and pairwise moments"
    p = model_probs(h_fit, J_fit)
    assert np.abs(W.T @ p - rates).max() < 3e-3
def test_tail():
    "the pairwise model misses the shape of the synchrony distribution: too few bins with 4 to 6 neurons active, too many with 9 or 10"
    assert mid_emp / mid_pair > 1.3 and top_pair / top_emp > 3

test_fit()
test_tail()
print("All checks passed")

### Exercise: Part B: loopy belief propagation

Complete the parallel message update on a 6-node cycle of binary units: each new message from i to j is the edge potential applied to i's evidence times the message into i from its other neighbour. Iterate 200 times, form beliefs, and compare with enumeration for coupling strengths 0.3 and 1.5.

In [ ]:
n6 = 6
cyc = [(i, (i + 1) % n6) for i in range(n6)]
nb6 = {i: [(i - 1) % n6, (i + 1) % n6] for i in range(n6)}
gB = np.random.default_rng(6261)
phi6 = np.exp(gB.normal(0, 0.8, (n6, 2)))

def exact(psi):
    m = np.zeros((n6, 2))
    for xs in itertools.product((0, 1), repeat=n6):
        w = np.prod([phi6[i, xs[i]] for i in range(n6)]) * np.prod([psi[xs[a], xs[b]] for a, b in cyc])
        for i in range(n6):
            m[i, xs[i]] += w
    return m / m.sum(1, keepdims=True)

def loopy(psi, iters=200):
    msg = {(i, j): np.ones(2) / 2 for i in range(n6) for j in nb6[i]}
    for _ in range(iters):
        new = {}
        for (i, j) in msg:
            k = [x for x in nb6[i] if x != j][0]
            m = ...
            new[(i, j)] = m / m.sum()
        msg = new
    b = np.array([phi6[i] * np.prod([msg[(k, i)] for k in nb6[i]], axis=0) for i in range(n6)])
    return b / b.sum(1, keepdims=True)

err = {}
for c in (0.3, 1.5):
    psi = np.exp(c * np.array([[1.0, -1.0], [-1.0, 1.0]]))
    err[c] = np.abs(loopy(psi) - exact(psi)).max()
    print(f"coupling {c}: largest error of loopy beliefs {err[c]:.4f}")

In [ ]:
import numpy as np
def test_weak():
    "with weak couplings, loopy beliefs are within 0.02 of the exact marginals"
    assert err[0.3] < 0.02
def test_strong():
    "with strong couplings, the error grows"
    assert err[1.5] > err[0.3] and err[1.5] > 0.03

test_weak()
test_strong()
print("All checks passed")